 ### Zoteroize and Obsidianize a Perplexity Dialogue



 In a Perplexity dialogue copied to the clipboard by the perplexity copy button and then saved to a file, replace

 the citation numbers with matching Obsidian literature note or Zotero item links

In [1]:
import re
import pathlib as pl
import sys
from collections import defaultdict
import pandas as pd
from icecream import ic

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import link_perplexity_zotero as lpz

%load_ext autoreload
%autoreload 2


In [ ]:

citenum_url_link_re = re.compile(r'\[(?P<orig>\d+)\]\((?P<url>https?://[^\)]+)\)')
sources_citenum_links_re = re.compile(r'\((?P<orig>\d+)\)\((?P<url>https?://[^\)]+)\)')
citenum_plain_re = re.compile(r'\[(?P<num>\d+)\]')

def dedup_citenums_to_urls(num_url_pairs: list[tuple[str, str]], verbose:) -> pd.DataFrame:
    """Return a dataframe showing remapping when >1 citenums map to the same URL."""
    url_to_citenums = defaultdict(list)
    for num, url in num_url_pairs:
        url_to_citenums[url].append(num)
    
    # Create new citation numbers if there are duplicates
    new_cite_num = 1
    lut = []
    for url, nums in url_to_citenums.items():
        if (nDups := len(nums)) > 1:
            print(f'URL has {nDups} dups: {nums=}, {url=}')
        for num in nums:
            lut.append({'orig_num': num, 'new_num': str(new_cite_num), 'url': url})
        
        new_cite_num += 1
    
    return pd.DataFrame(lut).set_index(['orig_num'])

def split_body_source(perplexity_file: pl.Path):
    """Replace links in standard Perplexity (saved clipboard) output with links 
    to Zotero items or Obsidian lit notes. """    
    
    content = perplexity_file.read_text(encoding='utf-8')
    section_parts = content.split("\nCitations:\n", 1)
    if len(section_parts) < 2:
        print("Missing citations")
        body, citations = section_parts, ""
    else:
        body, citations = section_parts

    # Reassign body cite numbers if duplicate URLs are found in the sources
    # TODO: replace regexp with citenum_plain_re ? that one is missing the trailing S+
    source_matches = list(re.finditer(r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)', citations, flags=re.M))
    num_url_pairs = [(match.group('num'), match.group('url')) for match in source_matches]
    #ic(num_url_pairs)
    citenums_to_url = dedup_citenums_to_urls(num_url_pairs)
    
    display(citenums_to_url)

    oldnum_to_new = citenums_to_url.new_num.to_dict() # global for replace_body_citenum()
    def replace_body_citenum(match):
        """Replace a cite number in the body with a new cite number.  This is used to renumber
        the body if duplicate cite numbers are found in the source list."""
        return f'[{oldnum_to_new[match.group("num")]}]'

    body = re.sub(citenum_plain_re, replace_body_citenum, body)
    
    return body, citenums_to_url

def relink_chunks(body: str, citenums_to_url: pd.DataFrame) -> tuple[str, str]:
    """Replaces body links with Zotero or Obsidian links, and returns the relinked body and sources."""
    def make_relinks_from_source(cite_num: str, doc_url: str) -> str:
        """Returns what a relinked citation would look like if present in the body,
        given a source part citation number and url.  Also appends to the global list, 
        relinked_sources, a relinked source part link.  Expects the global set, body_cite_nums."""
        
        numbered_link = f"[{cite_num}]({doc_url})"
        if zotero_item := relinker.find_zotero_item_via_url(doc_url):
            body_link = relinker.create_obsidian_or_zotero_link(zotero_item)
            relinked_sources.append(f'({numbered_link}) **{body_link}**')
        else:
            body_link = f"=={numbered_link}==" # mark it as "not in zotero"
            source_line = f'({numbered_link}) {doc_url}'
            source_line = f'=={source_line} ==' if cite_num in body_cite_nums else source_line
            relinked_sources.append(source_line)
            
        return body_link

    # get the map from citenums, in case they had to be redone    
    new_num_to_url = citenums_to_url.set_index('new_num').url.to_dict()
    
    # globals for make_relinks_from_source()
    body_cite_nums = set(re.findall(citenum_plain_re, body))
    relinked_sources = []
    relinker = lpz.ZoteroLinkConverter()

    # compute links to zotero and obsidian, when possible
    source_num_to_link = {num: make_relinks_from_source(num, url)
                          for num, url in new_num_to_url.items() }
    # replace the cite numbers with new links
    body_relinked = re.sub(citenum_plain_re, 
                           lambda m: f' {source_num_to_link.get(m.group("num"))}', body)
    # relinked sources were stored in this global
    sources_relinked = "\n".join(relinked_sources)
    
    return body_relinked, sources_relinked

def relink_perplexity_export(perplexity_file: pl.Path, relinked_file: pl.Path) -> None:
    body, citenums_to_url = split_body_source(perplexity_file)
    body_relinked, sources_relinked = relink_chunks(body, citenums_to_url)
    relinked_file.write_text(f'# Response\n{body_relinked}\n# Citations\n{sources_relinked}', encoding='utf-8')

In [54]:
#perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
perplexity_dialog_file = pl.Path(r'C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/merge_chats_perplex/PerPlexPro.md') # >1 for one URL
output_file = rfw.refwrangle_test_dir / 'tmp' / "tmp_new_cites_perplexity_example.md"
print(f'{perplexity_dialog_file=}\n-->\n{output_file=}')

relink_perplexity_export(perplexity_dialog_file, output_file)
print('Done.')

perplexity_dialog_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/merge_chats_perplex/PerPlexPro.md')
-->
output_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/tmp/tmp_new_cites_perplexity_example.md')
URL has 2 dups: nums=['4', '54'], url='https://globalaffairs.org/commentary-and-analysis/blogs/brazils-systemic-mistrust-elections-and-democracy'


,new_num,url
orig_num,,
1,1,https://en.wikipedia.org/wiki/Right-wing_populism
2,2,https://www.politico.eu/article/mapped-europe-...
3,3,https://www.npr.org/2024/06/09/nx-s1-4997712/f...
4,4,https://globalaffairs.org/commentary-and-analy...
54,4,https://globalaffairs.org/commentary-and-analy...
...,...,...
77,76,https://rioonwatch.org/?p=72542
78,77,https://www.populismstudies.org/chega-emerges-...
79,78,https://www.american.edu/sis/centers/transatla...


Reading from cache.
Done.


### Test merging

In [4]:
tmpdir = rfw.refwrangle_test_dir / 'tmp'
tmpdir.mkdir(parents=True, exist_ok=True)

datdir = rfw.refwrangle_test_dir / 'dat' / 'merge_chats_perplex'
datdir.mkdir(parents=True, exist_ok=True)

chat_files = list(datdir.glob('*.md'))
chat_files

merged_output_file = tmpdir / 'tmp_stock_perplexy_merged.md'

In [9]:
# get the sources from all docs to be merged
all_bodies, all_citenums_to_url = [], []
for chat_file in chat_files:
    body, citenums_to_url = split_body_source(chat_file)
    all_bodies.append(body)
    all_citenums_to_url.append(citenums_to_url)

URL has 2 dups: nums=['6', '41'], url='https://dgap.org/en/research/publications/right-wing-populists-would-win-poland-today'
URL has 2 dups: nums=['7', '67'], url='https://www.pewresearch.org/global/2024/12/11/global-elections-in-2024-what-we-learned-in-a-year-of-political-disruption/'
URL has 2 dups: nums=['9', '44'], url='https://foreignpolicy.com/2023/12/26/right-wing-populism-are-set-to-sweep-the-west-in-2024/'
URL has 2 dups: nums=['10', '50'], url='https://www.populismstudies.org/populism-in-2023-the-year-in-review/'
URL has 2 dups: nums=['11', '64'], url='https://carnegieendowment.org/posts/2022/09/how-a-far-right-victory-in-italy-might-ripple-through-the-eu?lang=en'
URL has 2 dups: nums=['12', '42'], url='https://dcubrexitinstitute.eu/2023/10/elections-in-poland-bring-the-end-of-right-wing-populist-rule/'
URL has 2 dups: nums=['4', '54'], url='https://globalaffairs.org/commentary-and-analysis/blogs/brazils-systemic-mistrust-elections-and-democracy'


In [ ]:
all_citenums_to_url

In [ ]:
#    body_relinked, sources_relinked = relink_chunks(body, citenums_to_url)

# find all the doc citations for each unique URL
url_to_citenums = defaultdict(list)
print(url_to_citenums)
for docIx, citenums_to_url in enumerate(all_citenums_to_url):
    for url, num in citenums_to_url.items():
        url_to_citenums[url].append(dict(orig_num=num, docIx=docIx))

# create new citenums for a combined document with a combined sources section
new_cite_num = 1
lut = []
for url, nums in url_to_citenums.items():
    for info in nums:
        lut.append({'url': url, 'new_cite_num': str(new_cite_num)} | info)
    new_cite_num += 1

lut = pd.DataFrame(lut).set_index(['docIx', 'orig_num'])
all_new_cite_nums = lut.new_cite_num.unique()

# make single body with cite numbers replaced by combined cite numbers

body_cites_not_in_sources = []
def replace_link_num(m):
    orig_citenum = m.group('orig')
    try:
        num = citenums_to_url[orig_citenum]
    except:
        chat_file=chat_files[docIx]
        print(f"Missing source for {orig_citenum} in {chat_file}")
        info = dict(orig_cite_num=orig_citenum, chat_file=chat_file)
        body_cites_not_in_sources.append(info)
        num = orig_citenum

    return f"[{num}]({m.group('url')})"



concat_bodies = ""
concat_sources = ""
for docIx, body in enumerate(all_bodies):
    # relink body with old citenums
    source_matches = citenums_to_url[docIx]
    url_to_source_nums = all_url_to_source_nums[docIx]
    body_relinked, sources_relinked = relink_chunks(body, source_matches, citenums_to_url)
    
    citenums_to_url =lut.loc[docIx].new_cite_num.to_dict()
    body_re_relinked = re.sub(citenum_url_link_re, replace_link_num, body_relinked)
    concat_bodies += f'# {chat_files[docIx].name}\n{body_re_relinked}\n'
    sources_re_relinked = re.sub(citenum_url_link_re, replace_link_num, sources_relinked)
    concat_sources += f'{sources_re_relinked}\n'

In [ ]:
import numpy as np
len(concat_sources.split('\n')), len(np.unique(concat_sources.split('\n')))

concat_sources_unique = list(set(concat_sources.split('\n')))
#sorted_strings = sorted(concat_sources_unique, key=lambda x: int(re.search(r'\((\d+)\]', x).group(1)))
#sorted_strings

# Function to extract the number inside [num]
def extract_number(s):
    match = re.search(r'\[(\d+)\]', s)
    return int(match.group(1)) if match else None  # Handle cases without [num]

# Sort the list using the extracted number as key
merged_sources = "\n".join(sorted(concat_sources_unique, key=extract_number))

ic(merged_output_file)
merged_output_file.write_text(f'# Responses\n{concat_bodies}\n# Citations\n{merged_sources}', encoding='utf-8')
print('Done.')

In [ ]:
import re

citations = """
[1] https://example.com/article1
[2] https://example.com/article2
[3] https://example.com/article3
"""

# Find matches using re.finditer
source_matches = list(re.finditer(r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)', citations, flags=re.M))
num_url_pairs = [(match.group('num'), match.group('url')) for match in source_matches]

# Iterate through num_url_pairs
for num, url in num_url_pairs:
    print(f"Number: {num}, URL: {url}")


Number: 1, URL: https://example.com/article1
Number: 2, URL: https://example.com/article2
Number: 3, URL: https://example.com/article3
